# SpaceX Falcon 9 - Data Collection via Web Scraping

**Note on execution:** this notebook scrapes the Wikipedia "List of Falcon 9 and Falcon Heavy launches" page to fill in launch records the API doesn't fully capture (particularly for cross-checking booster landing outcomes). The sandbox this project's other notebooks were finalized in cannot reach Wikipedia directly, so the cells below are provided as correct, ready-to-run code. Running this on a machine with normal internet access reproduces `spacex_web_scraped.csv`, the file the Data Wrangling notebook actually loads and cleans.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import unicodedata

static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
response = requests.get(static_url)
print(response.status_code)

### Parse the page and grab every launch table

In [ ]:
soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title.string)

html_tables = soup.find_all('table', class_='wikitable plainrowheaders collapsible')
print(f"Found {len(html_tables)} launch tables")

### Helper functions to clean each cell's text

In [ ]:
def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass = mass[0:mass.find("kg") + 2]
    else:
        new_mass = 0
    return new_mass

def extract_column_from_header(row):
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = ' '.join(row.contents)
    if not column_name.strip().isdigit():
        column_name = column_name.strip()
        return column_name

### Extract clean column names from the first table's header row

In [ ]:
first_launch_table = html_tables[2]
column_names = []
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

print(column_names)

### Walk every launch table and row, building the flat dictionary that becomes the DataFrame

In [ ]:
launch_dict = dict.fromkeys(column_names)
del launch_dict['Date and time ( )']

launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

extracted_row = 0
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    for rows in table.find_all("tr"):
        if rows.th and rows.th.string:
            flight_number = rows.th.string.strip()
            flag = flight_number.isdigit()
        else:
            flag = False

        row = rows.find_all('td')
        if flag:
            extracted_row += 1
            launch_dict['Flight No.'].append(flight_number)

            datatimelist = date_time(row[0])
            launch_dict['Date'].append(datatimelist[0].strip(','))
            launch_dict['Time'].append(datatimelist[1])

            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.string if row[1].a else ''
            launch_dict['Version Booster'].append(bv)

            launch_site = row[2].a.string if row[2].a else ''
            launch_dict['Launch site'].append(launch_site)

            payload = row[3].a.string if row[3].a else ''
            launch_dict['Payload'].append(payload)

            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)

            orbit = row[5].a.string if row[5].a else ''
            launch_dict['Orbit'].append(orbit)

            customer = row[6].a.string if row[6].a else ''
            launch_dict['Customer'].append(customer)

            launch_outcome = list(row[7].strings)[0]
            launch_dict['Launch outcome'].append(launch_outcome)

            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)

print(f"Extracted {extracted_row} launch rows")

### Build the DataFrame and save it

In [ ]:
df = pd.DataFrame({k: pd.Series(v) for k, v in launch_dict.items()})
df.to_csv('spacex_web_scraped.csv', index=False)
print("Saved", df.shape[0], "rows to spacex_web_scraped.csv")
df.head()

## Summary

This notebook scrapes the Wikipedia Falcon 9/Falcon Heavy launch table directly with BeautifulSoup (no API involved), parsing each row into flight number, date, booster version, launch site, payload, payload mass, orbit, customer, launch outcome, and booster landing status. The resulting `spacex_web_scraped.csv` is the raw input to the Data Wrangling notebook.